In [42]:
# Ucitavanje podataka
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

rawData = "../../data/raw/dataset.csv"
processedDir = "../../data/processed"

outPath = os.path.join(processedDir, "ProcessedData.csv")

df = pd.read_csv(rawData)
df.shape

(13910, 131)

In [43]:
# Inicijalno sredjivanje (Izbacivanje neNumerickih kolona i bespotrebnih kolona)
df = df.drop(columns=["id"])

y1 = df["Class"]
y2 = df["V129"]
X = df.drop(columns=["Class"])
X = X.drop(columns=["V129"])

X = X.select_dtypes(include=[np.number])
X.shape

(13910, 128)

In [44]:
# NaN analiza
nan = X.isna().sum().sum()
nan

0

In [45]:
# Rad sa outlierima
LQ = 0.01
RQ = 0.99

clippedX = X.copy()

brojOdsecenih = 0

for col in clippedX.columns:
    leftQ = clippedX[col].quantile(LQ)
    rightQ = clippedX[col].quantile(RQ)

    original = clippedX[col]

    nL = (original < leftQ).sum()
    nR = (original > rightQ).sum()
    brojOdsecenih += nL + nR

    clippedX[col] = original.clip(leftQ, rightQ)

brojOdsecenih

35840

In [46]:
# Standardizacija
scaler = RobustScaler(quantile_range=(5, 95))
scaledX = scaler.fit_transform(clippedX)

In [47]:
# Cuvanje pretprocesiranog skupa
processedData = pd.DataFrame(
    scaledX,
    columns=X.columns
)

if y1 is not None:
    processedData["Class"] = y1.values
    
if y2 is not None:
    processedData["V129"] = y2.values

processedData.to_csv(outPath, index=False)
processedData.shape

(13910, 130)

In [48]:
############################################################################

In [49]:
# Za seminarski 

# Koncentracija Gasa
if "V129" in processedData.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(processedData["V129"], bins=20, kde=False)
    plt.title("Koncentracija Gasa (V129)")
    plt.xlabel("Nivo koncentracije")
    plt.ylabel("Frekvencija")
    plt.tight_layout()
    plt.savefig(f"{processedDir}/HistogramKoncentracijeGasa.png", dpi=200)
    plt.close()

# PCA bez klasa
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(scaledX)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], s=5, alpha=0.6)
plt.title("PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(f"{processedDir}/PCA2D.png", dpi=200)
plt.close()

# Korelacije
corr = X.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap="coolwarm", square=True)
plt.title("Korelaciona heatmapa")
plt.tight_layout()
plt.savefig(f"{processedDir}/Korelacije.png", dpi=200)
plt.close()